# Vascular Network Experiment Notebook

Interactive testing of **Space Colonization** and **ODC (Optimized Directed Colonization)** backends.

No GUI required — everything runs from pure Python.

## 1. Setup & Imports

In [ ]:
import sys, os
from pathlib import Path

ROOT = str(Path(".").resolve().parent)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from test.space_colonization_runner import run_space_colonization, run_space_colonization_dual_tree
from test.odc_runner import run_odc
from test.notebook_utils import (
    plot_network_2d,
    plot_network_3d,
    print_stats,
    compare_networks,
    compare_stats_table,
    network_to_dataframe,
    save_network_json,
)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 2. Space Colonization
### 2a. Single-inlet (cylinder domain)

In [ ]:
sc_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "domain_center": [0.0, 0.0, 0.0],

    "inlet_position": [0.0, 0.0, 0.005],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",

    "num_attractors": 2000,
    "attraction_distance": 0.012,
    "kill_distance": 0.003,
    "step_size": 0.002,
    "max_iterations": 500,
    "max_steps": 200,
    "branch_angle_deg": 35.0,
    "directional_bias": 0.75,
    "max_deviation_deg": 35.0,

    "encourage_bifurcation": True,
    "max_children_per_node": 2,
    "bifurcation_probability": 0.65,
    "min_attractions_for_bifurcation": 4,
    "bifurcation_angle_threshold_deg": 50.0,

    "min_radius": 0.0001,
    "taper_factor": 0.93,

    "progress": True,
    "kdtree_rebuild_tip_every": 1,
    "kdtree_rebuild_all_nodes_every": 10,
    "stall_steps_per_inlet": 10,
    "interleaving_strategy": "round_robin",

    "check_collisions": True,
    "collision_clearance": 0.0002,
    "collision_merge_distance": 0.0003,

    "seed": 42,
    "num_outlets": 50,
}

sc_net, sc_stats = run_space_colonization(sc_params)
print_stats(sc_stats, "Space Colonization -- single inlet")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(sc_net, projection=proj, ax=ax, title=f"SC single-inlet ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_net, title="SC Single-Inlet 3D")
fig3d.show()

### 2b. Multi-inlet (blended mode)

In [ ]:
sc_multi_params = {
    **sc_params,
    "inlets": [
        {"position": [0.003, 0.0, 0.005], "radius": 0.0008},
        {"position": [-0.003, 0.0, 0.005], "radius": 0.0008},
        {"position": [0.0, 0.003, 0.005], "radius": 0.0008},
    ],
    "multi_inlet_mode": "blended",
    "multi_inlet_blend_sigma": 0.0,
    "max_inlets": 10,
    "num_attractors": 2000,
    "seed": 123,
}

sc_multi_net, sc_multi_stats = run_space_colonization(sc_multi_params)
print_stats(sc_multi_stats, "SC -- blended multi-inlet")
plot_network_2d(sc_multi_net, title="SC blended multi-inlet (xy)")
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_multi_net, title="SC blended multi-inlet 3D")
fig3d.show()

### 2c. Multi-inlet (partitioned_xy mode)

In [ ]:
sc_part_params = {
    **sc_multi_params,
    "multi_inlet_mode": "partitioned_xy",
    "partitioned_directional_bias": 1.0,
    "partitioned_max_deviation_deg": 30.0,
    "partitioned_cone_angle_deg": 30.0,
    "partitioned_cylinder_radius": 0.001,
    "seed": 123,
}

sc_part_net, sc_part_stats = run_space_colonization(sc_part_params)
print_stats(sc_part_stats, "SC -- partitioned_xy multi-inlet")
plot_network_2d(sc_part_net, title="SC partitioned_xy (xy)")
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_part_net, title="SC partitioned_xy 3D")
fig3d.show()

### 2d. Multi-inlet (forest mode)

In [ ]:
sc_forest_params = {
    **sc_multi_params,
    "multi_inlet_mode": "forest",
    "seed": 123,
}

sc_forest_net, sc_forest_stats = run_space_colonization(sc_forest_params)
print_stats(sc_forest_stats, "SC -- forest multi-inlet")
plot_network_2d(sc_forest_net, title="SC forest mode (xy)")
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_forest_net, title="SC forest multi-inlet 3D")
fig3d.show()

### 2e. Parameter sweep -- num_attractors

In [ ]:
sweep_results = {}
for n_att in [500, 1000, 2000, 4000]:
    p = {**sc_params, "num_attractors": n_att, "seed": 42}
    net, stats = run_space_colonization(p)
    sweep_results[f"attractors={n_att}"] = (net, stats)

compare_networks(sweep_results, projection="xy")

In [ ]:
compare_stats_table({k: v[1] for k, v in sweep_results.items()})

In [ ]:
for label, (net, stats) in sweep_results.items():
    fig3d = plot_network_3d(net, title=f"SC {label} 3D")
    fig3d.show()

### 2f. Dual tree (blended, no merge-on-collision)

In [ ]:
sc_dual_params = {
    **sc_params,
    "inlets": [
        {"position": [0.0, 0.0, 0.005], "radius": 0.001},
        {"position": [0.0, 0.0, -0.005], "radius": 0.001},
    ],
    "multi_inlet_mode": "blended",
    "multi_inlet_blend_sigma": 0.0,
    "num_attractors": 1500,
    "max_inlets": 10,
    "seed": 42,
    "progress": True,
}

sc_dual_net, sc_dual_stats = run_space_colonization(sc_dual_params)
print_stats(sc_dual_stats, "SC Dual Tree (blended, no merge-on-collision)")
plot_network_2d(sc_dual_net, title="SC Dual Tree - blended (xy)")
plt.show()

In [ ]:
fig3d = plot_network_3d(sc_dual_net, title="SC Dual Tree - blended 3D")
fig3d.show()

## 3. Optimized Directed Colonization (ODC)

ODC extends space colonization with **hierarchical tissue targeting** and **Murray's law post-propagation**.
Different tissue point distributions produce unique branching structures.
Hierarchical ordering directs growth: high-priority regions are reached first.

### 3a. ODC with auto-generated hierarchical levels

Auto-generates 3 priority levels: deep interior -> mid-range -> filler.
The network grows toward the center first, then expands outward.

In [ ]:
odc_params = {
    "domain_type": "cylinder",
    "domain_radius": 0.005,
    "domain_height": 0.010,
    "inlet_position": [0.0, 0.0, 0.005],
    "inlet_radius": 0.001,
    "vessel_type": "arterial",
    "influence_radius": 0.015,
    "kill_radius": 0.003,
    "step_size": 0.005,
    "max_steps": 500,
    "bifurcation_probability": 0.7,
    "max_children_per_node": 2,
    "taper_factor": 0.95,
    "auto_n_levels": 3,
    "auto_points_per_level": 200,
    "apply_murray": True,
    "murray_exponent": 3.0,
    "terminal_radius": 0.0003,
    "seed": 42,
}

odc_net, odc_stats = run_odc(odc_params)
print_stats(odc_stats, "ODC -- auto hierarchical levels")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(odc_net, projection=proj, ax=ax, title=f"ODC auto ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(odc_net, title="ODC Auto-Hierarchical 3D")
fig3d.show()

### 3b. Seed sweep -- unique structures from different tissue distributions

Each seed creates a different random tissue distribution, producing
unique but biologically valid branching patterns.

In [ ]:
odc_seed_results = {}
for seed in [1, 42, 123, 999]:
    p = {**odc_params, "seed": seed}
    net, stats = run_odc(p)
    odc_seed_results[f"seed={seed}"] = (net, stats)

compare_networks(odc_seed_results, projection="xy")
compare_stats_table({k: v[1] for k, v in odc_seed_results.items()})

In [ ]:
for label, (net, stats) in odc_seed_results.items():
    fig3d = plot_network_3d(net, title=f"ODC {label} 3D")
    fig3d.show()

### 3c. Explicit tissue levels -- directing growth via ordering

Define custom tissue levels to control WHERE the network grows first.
Level 0 (highest priority) is reached before level 1, which is reached before level 2.

In [ ]:
rng = np.random.default_rng(42)

level_0_pts = np.column_stack([
    rng.normal(0, 0.001, 50),
    rng.normal(0, 0.001, 50),
    rng.uniform(-0.004, -0.002, 50),
])

level_1_pts = np.column_stack([
    rng.uniform(-0.003, 0.003, 100),
    rng.uniform(-0.003, 0.003, 100),
    rng.uniform(-0.003, 0.001, 100),
])

level_2_pts = np.column_stack([
    rng.uniform(-0.004, 0.004, 200),
    rng.uniform(-0.004, 0.004, 200),
    rng.uniform(-0.004, 0.004, 200),
])

odc_explicit_params = {
    **odc_params,
    "tissue_levels": [
        {"priority": 0, "points": level_0_pts.tolist(), "label": "deep core",
         "weight": 1.0, "coverage_threshold": 0.8},
        {"priority": 1, "points": level_1_pts.tolist(), "label": "mid-range",
         "weight": 0.6, "coverage_threshold": 0.7},
        {"priority": 2, "points": level_2_pts.tolist(), "label": "outer filler",
         "weight": 0.3, "coverage_threshold": 0.5},
    ],
    "augment_with_filler": False,
    "seed": 42,
}

odc_explicit_net, odc_explicit_stats = run_odc(odc_explicit_params)
print_stats(odc_explicit_stats, "ODC -- explicit 3-level")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, proj in zip(axes, ["xy", "xz", "yz"]):
    plot_network_2d(odc_explicit_net, projection=proj, ax=ax, title=f"ODC explicit ({proj})")
plt.tight_layout()
plt.show()

In [ ]:
fig3d = plot_network_3d(odc_explicit_net, title="ODC Explicit Levels 3D")
fig3d.show()

### 3d. Tissue distribution comparison -- clustered vs spread

Tightly clustered level-0 points produce focused branching; spread-out points produce wider trees.

In [ ]:
rng2 = np.random.default_rng(7)

clustered_l0 = np.column_stack([
    rng2.normal(0.002, 0.0005, 80),
    rng2.normal(-0.002, 0.0005, 80),
    rng2.normal(-0.003, 0.0005, 80),
])

spread_l0 = np.column_stack([
    rng2.uniform(-0.004, 0.004, 80),
    rng2.uniform(-0.004, 0.004, 80),
    rng2.uniform(-0.004, 0.004, 80),
])

odc_clustered_params = {
    **odc_params,
    "tissue_levels": [
        {"priority": 0, "points": clustered_l0.tolist(), "label": "cluster",
         "weight": 1.0, "coverage_threshold": 0.8},
    ],
    "augment_with_filler": True,
    "filler_n_points": 300,
    "seed": 7,
}

odc_spread_params = {
    **odc_params,
    "tissue_levels": [
        {"priority": 0, "points": spread_l0.tolist(), "label": "spread",
         "weight": 1.0, "coverage_threshold": 0.8},
    ],
    "augment_with_filler": True,
    "filler_n_points": 300,
    "seed": 7,
}

odc_clustered_net, odc_clustered_stats = run_odc(odc_clustered_params)
odc_spread_net, odc_spread_stats = run_odc(odc_spread_params)

compare_networks({
    "Clustered L0": (odc_clustered_net, odc_clustered_stats),
    "Spread L0": (odc_spread_net, odc_spread_stats),
}, projection="xy")

compare_stats_table({
    "Clustered": odc_clustered_stats,
    "Spread": odc_spread_stats,
})

In [ ]:
fig3d_c = plot_network_3d(odc_clustered_net, title="ODC Clustered 3D")
fig3d_c.show()

fig3d_s = plot_network_3d(odc_spread_net, title="ODC Spread 3D")
fig3d_s.show()

## 4. Head-to-Head: Space Colonization vs ODC

Compare both algorithms on the same cylinder domain.
SC uses attractor-driven growth; ODC adds hierarchical tissue ordering + Murray radii.

In [ ]:
compare_networks({
    "Space Colonization": (sc_net, sc_stats),
    "ODC auto": (odc_net, odc_stats),
    "ODC explicit": (odc_explicit_net, odc_explicit_stats),
}, projection="xz")

In [ ]:
compare_stats_table({
    "SC": sc_stats,
    "ODC auto": odc_stats,
    "ODC explicit": odc_explicit_stats,
})

In [ ]:
fig3d_sc = plot_network_3d(sc_net, title="SC 3D (head-to-head)")
fig3d_sc.show()

fig3d_odc = plot_network_3d(odc_net, title="ODC auto 3D (head-to-head)")
fig3d_odc.show()

fig3d_odc_e = plot_network_3d(odc_explicit_net, title="ODC explicit 3D (head-to-head)")
fig3d_odc_e.show()

### 4b. Branching characteristics

Compare segment length and radius distributions side by side.

In [ ]:
df_sc = network_to_dataframe(sc_net)
df_odc = network_to_dataframe(odc_net)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(df_sc["length_m"] * 1000, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
axes[0, 0].set_xlabel("Segment length (mm)")
axes[0, 0].set_title("SC -- segment lengths")

axes[0, 1].hist(df_odc["length_m"] * 1000, bins=30, color="darkorange", edgecolor="white", alpha=0.8)
axes[0, 1].set_xlabel("Segment length (mm)")
axes[0, 1].set_title("ODC -- segment lengths")

axes[1, 0].hist(df_sc["mean_radius_m"] * 1e6, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
axes[1, 0].set_xlabel("Mean radius (um)")
axes[1, 0].set_title("SC -- vessel radii")

axes[1, 1].hist(df_odc["mean_radius_m"] * 1e6, bins=30, color="darkorange", edgecolor="white", alpha=0.8)
axes[1, 1].set_xlabel("Mean radius (um)")
axes[1, 1].set_title("ODC -- vessel radii (Murray-propagated)")

plt.tight_layout()
plt.show()

## 5. Segment-Level Analysis

In [ ]:
df = network_to_dataframe(sc_net)
df.head(10)

In [ ]:
df["length_mm"] = df["length_m"] * 1000
df["mean_radius_um"] = df["mean_radius_m"] * 1e6

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df["length_mm"], bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Segment length (mm)")
axes[0].set_ylabel("Count")
axes[0].set_title("Segment length distribution")

axes[1].hist(df["mean_radius_um"], bins=30, color="indianred", edgecolor="white")
axes[1].set_xlabel("Mean radius (um)")
axes[1].set_ylabel("Count")
axes[1].set_title("Vessel radius distribution")

plt.tight_layout()
plt.show()

## 6. Export

In [ ]:
save_network_json(sc_net, "sc_network.json")
save_network_json(odc_net, "odc_network.json")
print("Networks saved to sc_network.json and odc_network.json")

## 7. Sandbox

Use cells below for your own experiments.